# 01 — Parse

Builds **LIB**, **CORPUS**, and **VOCAB** tables from the raw Studio Ghibli screenplay `.txt` files.

**OHCO**: `film_name → chunk_num → sent_num → token_num`  
A *chunk* is a single speaker utterance. Only dialogue with speaker attribution is retained;
stage directions, action lines, and narration are dropped to ensure cross-film comparability
(the 14 files range from rich full-screenplay format to bare SRT subtitles with no stage directions).

`speaker` is stored as a regular CORPUS column (not an OHCO level) because SRT-format
films carry no per-character attribution.

## Setup

In [26]:
import pandas as pd
import numpy as np
import re
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

DATA_DIR = 'kaggle-dataset'
OUTPUT_DIR = 'tables'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Film Metadata

- `start_line`: 0-indexed line where actual screenplay content begins (skips each file's header/preamble)
- `fmt`: which speaker-extraction strategy to apply
  - `dialogue` — character name on its own line, dialogue on following lines; handles `SPEAKER: text` and `SPEAKER    text` variants too
  - `timestamp` — subtitle-derived `[MM:SS]  SPEAKER: text` per line
  - `howls` — `Name    text` per line with indented continuation chunks
  - `srt` — raw SRT subtitles, no per-character attribution

In [27]:
OHCO = ['film_name', 'chunk_num', 'sent_num', 'token_num']

FILM_META = {
    'castle_in_the_sky':      {'director': 'Miyazaki', 'year': 1986, 'start_line': 93,  'fmt': 'dialogue',  'file': 'miyazaki_castle_in_the_sky'},
    'howls_moving_castle':    {'director': 'Miyazaki', 'year': 2004, 'start_line': 13,  'fmt': 'howls',     'file': 'miyazaki_howls_moving_castle'},
    'kikis_delivery_service': {'director': 'Miyazaki', 'year': 1989, 'start_line': 39,  'fmt': 'dialogue',  'file': 'miyazaki_kikis_delivery_service'},
    'my_neighbor_totoro':     {'director': 'Miyazaki', 'year': 1988, 'start_line': 59,  'fmt': 'dialogue',  'file': 'miyazaki_my_neighbor_totoro'},
    'nausicaa':               {'director': 'Miyazaki', 'year': 1984, 'start_line': 44,  'fmt': 'dialogue',  'file': 'miyazaki_nausicaa'},
    'ponyo':                  {'director': 'Miyazaki', 'year': 2008, 'start_line': 28,  'fmt': 'timestamp', 'file': 'miyazaki_ponyo'},
    'porco_rosso':            {'director': 'Miyazaki', 'year': 1992, 'start_line': 27,  'fmt': 'dialogue',  'file': 'miyazaki_porco_rosso'},
    'princess_mononoke':      {'director': 'Miyazaki', 'year': 1997, 'start_line': 19,  'fmt': 'dialogue',  'file': 'miyazaki_princess_mononoke'},
    'spirited_away':          {'director': 'Miyazaki', 'year': 2001, 'start_line': 0,   'fmt': 'dialogue',  'file': 'miyazaki_spirited_away'},
    'the_wind_rises':         {'director': 'Miyazaki', 'year': 2013, 'start_line': 29,  'fmt': 'timestamp', 'file': 'miyazaki_the_wind_rises'},
    'grave_of_the_fireflies': {'director': 'Takahata', 'year': 1988, 'start_line': 132, 'fmt': 'dialogue',  'file': 'takahata_grave_of_the_fireflies'},
    'only_yesterday':         {'director': 'Takahata', 'year': 1991, 'start_line': 118, 'fmt': 'dialogue',  'file': 'takahata_only_yesterday'},
    'pom_poko':               {'director': 'Takahata', 'year': 1994, 'start_line': 0,   'fmt': 'srt',       'file': 'takahata_pom_poko'},
    'princess_kaguya':        {'director': 'Takahata', 'year': 2013, 'start_line': 0,   'fmt': 'srt',       'file': 'takahata_princess_kaguya'},
}

## Universal Noise Removal

Strip translator footnotes and music symbols from raw text before format-specific parsing.

In [28]:
def remove_noise(text):
    """Strip universal noise that appears in multiple files."""
    # {translator footnote blocks} — may span multiple lines (only_yesterday)
    text = re.sub(r'\{[^}]*\}', '', text, flags=re.DOTALL)
    # [ALT/NOTE/LT/ET: ...] inline translator annotations (grave_of_the_fireflies)
    text = re.sub(r'\[(?:ALT|NOTE|LT|ET)[^\]]*\]', '', text, flags=re.IGNORECASE)
    # <hidden reference> angle-bracket annotations (only_yesterday)
    text = re.sub(r'<[^>]{1,60}>', '', text)
    # Music note symbols
    text = re.sub(r'[\u266a\u266b]', '', text)
    return text

## Format-Specific Utterance Extractors

Each extractor returns a list of `(speaker, text)` tuples representing speaker-attributed dialogue chunks.
Stage directions, action lines, and unattributed narration are dropped.

**Key design note**: `extract_dialogue` uses **line-by-line state tracking** rather than chunk splitting.
Chunk splitting failed because:
- Many files (totoro, nausicaa, kiki, spirited_away) have `SPEAKER` name headers with no blank line
  between consecutive speakers, so they collapse into one chunk attributed to the first speaker
- Some chunks start with a stage direction `(...)` followed immediately by dialogue lines —
  dropping the whole chunk also drops all the dialogue inside it

Parentheticals like `(V.O.)` and `(CONT'D)` are stripped from speaker names before storing.

In [29]:
# ---------------------------------------------------------------------------
# SRT format  (pom_poko, princess_kaguya)
# ---------------------------------------------------------------------------
def extract_srt(text):
    """SRT has no per-character labels. Italic subtitles → NARRATOR; plain → UNKNOWN."""
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    # Full timestamp lines: "HH:MM:SS,mmm --> HH:MM:SS,mmm"
    text = re.sub(r'^\d{2}:\d{2}:\d{2},\d+ --> \d{2}:\d{2}:\d{2},\d+\s*$', '',
                  text, flags=re.MULTILINE)
    # Orphan half-timestamps (pom_poko has blocks like " 00:04:11,451" with leading space)
    text = re.sub(r'^\s*\d{2}:\d{2}:\d{2},\d+\s*$', '', text, flags=re.MULTILINE)

    utterances = []
    for block in re.split(r'\n\s*\n', text):
        block = block.strip()
        if not block:
            continue
        content_lines = [l.strip() for l in block.split('\n') if l.strip()]
        all_italic = all(re.match(r'^<i>', l) for l in content_lines)
        speaker = 'NARRATOR' if all_italic else 'UNKNOWN'
        clean = re.sub(r'<[^>]+>', '', block).strip()
        if clean:
            utterances.append((speaker, clean))
    return utterances


# ---------------------------------------------------------------------------
# Timestamp format  (ponyo, wind_rises)
# Format: [MM:SS]  SPEAKER: dialogue text
# ---------------------------------------------------------------------------
def extract_timestamp(text):
    """Each line is one utterance: [MM:SS]  SPEAKER: text."""
    text = re.sub(r'^─{3,}.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\[scene break[^\]]*\]\s*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE)

    utterances = []
    for line in text.split('\n'):
        line = line.strip()
        line = re.sub(r'^\[\d+:\d+\]\s+', '', line)
        m = re.match(r'^([A-Z][A-Z_]+):\s+(.+)$', line)
        if m:
            utterances.append((m.group(1), m.group(2)))
    return utterances


# ---------------------------------------------------------------------------
# Howl's Moving Castle format
# Lines: "Name    dialogue"  OR  "            continuation" (many leading spaces)
# ---------------------------------------------------------------------------
def extract_howls(text):
    """Line-by-line extraction; continuation lines (4+ leading spaces) go to last speaker."""
    text = re.sub(r'^\[(?:Scene|Action):.*?\]\s*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE | re.DOTALL)

    utterances = []
    last_speaker = None
    for line in text.split('\n'):
        m = re.match(r'^([A-Za-z][A-Za-z\s]+?)\s{4,}(.+)$', line)
        if m:
            last_speaker = m.group(1).strip()
            utterances.append((last_speaker, m.group(2).strip()))
        elif line.strip() and re.match(r'^\s{4,}', line) and last_speaker:
            utterances.append((last_speaker, line.strip()))
    return utterances


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def is_character_name(line):
    """Return True if line looks like a screenplay character name (short, no sentence punctuation)."""
    bare = re.sub(r'\s*\([^)]+\)\s*', '', line).strip()
    if not bare or len(bare) > 50:
        return False
    if re.search(r'[.!?,]', bare):
        return False
    if re.match(r"^[A-Z][A-Z\s'/\-\.]+$", bare):
        return True
    if re.match(r'^[A-Z][a-z]+(?:\s+[A-Za-z]+){0,3}$', bare):
        return True
    return False


# ---------------------------------------------------------------------------
# Dialogue format  (all films except howls, timestamp, srt)
#
# Line-by-line state machine: avoids losing dialogue that follows a stage-direction
# line within the same blank-line block, and handles consecutive "Speaker: text"
# lines with no blank separator between them.
# Parentheticals like (V.O.) and (CONT'D) are stripped from speaker names.
# ---------------------------------------------------------------------------
def extract_dialogue(text):
    """Line-by-line speaker detection; flush on new speaker or blank line."""
    text = re.sub(r'^[\-=+*~_]{3,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\d+\.\s*$', '', text, flags=re.MULTILINE)

    utterances = []
    cur_speaker = None
    cur_lines   = []

    def flush():
        nonlocal cur_speaker, cur_lines
        if cur_speaker and cur_lines:
            utterances.append((cur_speaker, ' '.join(cur_lines)))
        cur_speaker = None
        cur_lines   = []

    PAT_A = re.compile(r"^([A-Za-z][A-Za-z\s'/\-\.]*?)(?:\s*\([^)]+\))?:\s+(.+)$")
    PAT_B = re.compile(r"^([A-Z][A-Z\s'/\-]+):$")
    PAT_D = re.compile(r"^([A-Z][A-Z\s'/\-/]+)\s{4,}(.+)$")
    # LISTSERV email footers: "Subject:", "Date:", "Reply-To:" etc. at end of fan-script files.
    # Require ": " (colon + space) to avoid matching dialogue like "From yesterday?"
    FOOTER = re.compile(r'^(?:Subject|Date|Reply-To|Message-ID|Sender|Organization):\s+\S',
                        re.IGNORECASE)

    for line in text.split('\n'):
        stripped = line.strip()

        if not stripped:
            flush()
            continue

        # File-footer sentinel: email/LISTSERV header → stop
        if FOOTER.match(stripped):
            flush()
            break

        # Stage direction / action annotation → skip line, keep current speaker
        if stripped.startswith('(') or stripped.startswith('['):
            continue

        # Pattern A: "SPEAKER: dialogue"  (optional parenthetical before colon)
        m = PAT_A.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = [m.group(2).strip()]
            continue

        # Pattern D: "SPEAKER    dialogue"  (4+ spaces, no colon) — grave
        m = PAT_D.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = [m.group(2).strip()]
            continue

        # Pattern B: "SPEAKER:" alone — mononoke
        m = PAT_B.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = []
            continue

        # Pattern C: character name alone on a line — castle, porco, spirited_away, etc.
        # Strip (V.O.) / (CONT'D) parentheticals from the stored speaker name.
        if is_character_name(stripped):
            flush()
            cur_speaker = re.sub(r'\s*\([^)]+\)\s*', '', stripped).strip()
            cur_lines   = []
            continue

        # Continuation: append to current speaker
        if cur_speaker is not None:
            cur_lines.append(stripped)
        # else: unattributed narration/action → DROP

    flush()
    return utterances

## Parse Function

Dispatches to the right extractor, then sentence-tokenizes and POS-tags each utterance.

In [30]:
stemmer1 = PorterStemmer()
stemmer2 = SnowballStemmer("english")
stemmer3 = LancasterStemmer()

def coarse_pos(tag):
    """Map Penn Treebank tag to coarse group: n/v/j/r/x."""
    if tag.startswith('NN'): return 'n'
    if tag.startswith('VB'): return 'v'
    if tag.startswith('JJ'): return 'j'
    if tag.startswith('RB'): return 'r'
    return 'x'

EXTRACTORS = {
    'srt':        extract_srt,
    'timestamp':  extract_timestamp,
    'howls':      extract_howls,
    'dialogue':   extract_dialogue
}

def parse_film(film_name, meta):
    with open(f"{DATA_DIR}/{meta['file']}.txt", encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    text = ''.join(lines[meta['start_line']:])
    text = remove_noise(text)

    utterances = EXTRACTORS[meta['fmt']](text)

    rows = []
    for chunk_num, (speaker, dialogue) in enumerate(utterances):
        for sent_num, sent in enumerate(sent_tokenize(dialogue)):
            token_num = 0
            for token, tag in pos_tag(word_tokenize(sent)):
                term = re.sub(r'[\W_]+', '', token).lower()
                if not term:
                    continue
                rows.append((
                    film_name, chunk_num, sent_num, token_num,
                    token, term, speaker, tag, coarse_pos(tag)
                ))
                token_num += 1

    return rows

## Build CORPUS

In [31]:
all_rows = []
for film_name, meta in FILM_META.items():
    rows = parse_film(film_name, meta)
    all_rows.extend(rows)
    print(f"{film_name:<45} {len(rows):>7,} tokens")

CORPUS = pd.DataFrame(
    all_rows,
    columns=OHCO + ['token_str', 'term_str', 'speaker', 'pos', 'pos_group']
).set_index(OHCO)

print(f"\nTotal CORPUS tokens: {len(CORPUS):,}")
CORPUS.head(10)

castle_in_the_sky                               6,550 tokens
howls_moving_castle                             8,673 tokens
kikis_delivery_service                          6,898 tokens
my_neighbor_totoro                              3,565 tokens
nausicaa                                        7,165 tokens
ponyo                                           3,657 tokens
porco_rosso                                     6,804 tokens
princess_mononoke                               8,429 tokens
spirited_away                                   7,813 tokens
the_wind_rises                                  6,006 tokens
grave_of_the_fireflies                          4,260 tokens
only_yesterday                                  7,317 tokens
pom_poko                                       12,496 tokens
princess_kaguya                                 6,363 tokens

Total CORPUS tokens: 95,996


token_str term_str  speaker  \
film_name         chunk_num sent_num token_num                               
castle_in_the_sky 0         0        0                Ah       ah      MEN   
                  1         0        0               Wah      wah  CREWMAN   
                            1        0                It       it  CREWMAN   
                                     1                's        s  CREWMAN   
                                     2                 a        a  CREWMAN   
                                     3               gas      gas  CREWMAN   
                                     4              bomb     bomb  CREWMAN   
                            2        0                It       it  CREWMAN   
                                     1                's        s  CREWMAN   
                                     2                an       an  CREWMAN   

                                                pos pos_group  
film_name         chunk_num sent_num token_num                 
castle_in_the_sky 0         0        0           NN         n  
                  1         0        0           NN         n  
                            1        0          PRP         x  
                                     1          VBZ         v  
                                     2           DT         x  
                                     3           NN         n  
                                     4           NN         n  
                            2        0          PRP         x  
                                     1          VBZ         v  
                                     2           DT         x

## Build LIB

In [32]:
lib_rows = []
for film_name, meta in FILM_META.items():
    film_tokens = CORPUS.xs(film_name, level='film_name')
    lib_rows.append({
        'film_name': film_name,
        'director':  meta['director'],
        'year':      meta['year'],
        'n_chunks':  film_tokens.index.get_level_values('chunk_num').nunique(),
        'n_tokens':  len(film_tokens),
    })

LIB = pd.DataFrame(lib_rows).set_index('film_name')
LIB

,director,year,n_chunks,n_tokens
film_name,,,,
castle_in_the_sky,Miyazaki,1986,897,6550
howls_moving_castle,Miyazaki,2004,1353,8673
kikis_delivery_service,Miyazaki,1989,678,6898
my_neighbor_totoro,Miyazaki,1988,464,3565
nausicaa,Miyazaki,1984,848,7165
ponyo,Miyazaki,2008,817,3657
porco_rosso,Miyazaki,1992,665,6804
princess_mononoke,Miyazaki,1997,760,8429
spirited_away,Miyazaki,2001,895,7813


In [33]:
char_counts = CORPUS.reset_index().groupby('film_name')['token_str'].apply(
    lambda x: x.str.len().sum()
)
np.mean(char_counts)

np.float64(26559.928571428572)

## Build VOCAB

In [34]:
VOCAB = CORPUS['term_str'].value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB['p'] = VOCAB['n'] / VOCAB['n'].sum()
VOCAB['i'] = np.log2(1 / VOCAB['p'])
VOCAB['n_chars'] = VOCAB.index.str.len()

# Most frequent POS tag per term
VOCAB['max_pos'] = CORPUS[['term_str', 'pos']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB['max_pos_group'] = CORPUS[['term_str', 'pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

# Compute POS ambiguity
VOCAB['n_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack().count(1)
VOCAB['cat_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().to_frame('n').reset_index()\
    .groupby('term_str').pos_group.apply(lambda x: set(x))
VOCAB['n_pos'] = CORPUS[['term_str','pos']].value_counts().unstack().count(1)
VOCAB['cat_pos'] = CORPUS[['term_str','pos']].value_counts().to_frame('n').reset_index()\
    .groupby('term_str').pos.apply(lambda x: set(x))

stop_words = set(stopwords.words('english'))
VOCAB['stop'] = VOCAB.index.isin(stop_words)

VOCAB['stem_porter'] = VOCAB.index.map(stemmer1.stem)
VOCAB['stem_snowball'] = VOCAB.index.map(stemmer2.stem)
VOCAB['stem_lancaster'] = VOCAB.index.map(stemmer3.stem)

VOCAB = VOCAB.sort_index()
print(f"VOCAB size: {len(VOCAB):,} unique terms")
VOCAB

VOCAB size: 6,630 unique terms


,n,p,i,n_chars,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,stem_porter,stem_snowball,stem_lancaster
term_str,,,,,,,,,,,,,,
0,1,0.000010,16.550687,1,CD,x,1,{x},1,{CD},False,0,0,0
05,1,0.000010,16.550687,2,CD,x,1,{x},1,{CD},False,05,05,05
1,41,0.000427,11.193135,1,CD,x,1,{x},1,{CD},False,1,1,1
10,8,0.000083,13.550687,2,CD,x,1,{x},1,{CD},False,10,10,10
100,2,0.000021,15.550687,3,CD,x,1,{x},1,{CD},False,100,100,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zero,1,0.000010,16.550687,4,JJ,j,1,{j},1,{JJ},False,zero,zero,zero
zo,2,0.000021,15.550687,2,NN,n,1,{n},1,{NN},False,zo,zo,zo
zohsui,1,0.000010,16.550687,6,CD,x,1,{x},1,{CD},False,zohsui,zohsui,zohsu


In [35]:
# VOCAB.loc[(VOCAB.n_pos >= 2) & (VOCAB.max_pos.str.match('(VB|NN)'))].sample(10)

In [36]:
print("Stopwords:", int(VOCAB['stop'].sum()), f"({VOCAB['stop'].mean():.1%} of types)")

Stopwords: 146 (2.2% of types)


# Average Chars

In [41]:
from pathlib import Path

folder = Path("kaggle-dataset")

lengths = [len(f.read_text(encoding="utf-8")) for f in folder.glob("*.txt")]

avg_length = sum(lengths) / len(lengths)
print(avg_length)

70939.78571428571


## Corpus length

In [42]:
print(f"\nTotal CORPUS tokens: {len(CORPUS):,}")


Total CORPUS tokens: 95,996


## Sanity Checks

In [37]:
print("=== Tokens per film ===")
print(LIB[['director', 'year', 'n_chunks', 'n_tokens']].to_string())

print("\n=== Tokens by director ===")
print(LIB.groupby('director')['n_tokens'].agg(['sum', 'mean']).astype(int))

print("\n=== Top 20 terms by frequency ===")
print(VOCAB.sort_values('n', ascending=False).head(20)[['n', 'p', 'stop']])

=== Tokens per film ===
                        director  year  n_chunks  n_tokens
film_name                                                 
castle_in_the_sky       Miyazaki  1986       897      6550
howls_moving_castle     Miyazaki  2004      1353      8673
kikis_delivery_service  Miyazaki  1989       678      6898
my_neighbor_totoro      Miyazaki  1988       464      3565
nausicaa                Miyazaki  1984       848      7165
ponyo                   Miyazaki  2008       817      3657
porco_rosso             Miyazaki  1992       665      6804
princess_mononoke       Miyazaki  1997       760      8429
spirited_away           Miyazaki  2001       895      7813
the_wind_rises          Miyazaki  2013       976      6006
grave_of_the_fireflies  Takahata  1988       367      4260
only_yesterday          Takahata  1991       833      7317
pom_poko                Takahata  1994      1626     12496
princess_kaguya         Takahata  2013      1224      6363

=== Tokens by director ===
    

In [38]:
# Spot-check: verify start_line and speaker extraction worked correctly for each film
for film_name in FILM_META:
    sample = CORPUS.xs(film_name, level='film_name').head(3)
    speakers = sample['speaker'].unique()
    first_tokens = ' '.join(sample['token_str'].tolist())
    print(f"{film_name:<45} speakers={speakers}  first_tokens='{first_tokens}'")

castle_in_the_sky                             speakers=['MEN' 'CREWMAN']  first_tokens='Ah Wah It'
howls_moving_castle                           speakers=['Bessie']  first_tokens='Sophie We just'
kikis_delivery_service                        speakers=['RADIO']  first_tokens='Now for the'
my_neighbor_totoro                            speakers=['Satsuki' 'Father']  first_tokens='Father caramel Oh'
nausicaa                                      speakers=['YUPA']  first_tokens='Another village dead'
ponyo                                         speakers=['LISA']  first_tokens='Sousuke come right'
porco_rosso                                   speakers=['Porco Rosso' 'Phone voice']  first_tokens='Yeah Porco Rosso'
princess_mononoke                             speakers=['ASHITAKA' 'KAYA']  first_tokens='Yakkuru Ani-sama Hii-sama'
spirited_away                                 speakers=['CHIHIRO']  first_tokens='I ll miss'
the_wind_rises                                speakers=['CAPTION']  first

## Save Tables

In [39]:
LIB.to_csv(f'{OUTPUT_DIR}/LIB.csv')
CORPUS.to_csv(f'{OUTPUT_DIR}/CORPUS.csv')
VOCAB.to_csv(f'{OUTPUT_DIR}/VOCAB.csv')
print(f"Saved:")
print(f"  LIB    — {len(LIB)} films          → {OUTPUT_DIR}/LIB.csv")
print(f"  CORPUS — {len(CORPUS):,} tokens    → {OUTPUT_DIR}/CORPUS.csv")
print(f"  VOCAB  — {len(VOCAB):,} unique terms → {OUTPUT_DIR}/VOCAB.csv")

Saved:
  LIB    — 14 films          → tables/LIB.csv
  CORPUS — 95,996 tokens    → tables/CORPUS.csv
  VOCAB  — 6,630 unique terms → tables/VOCAB.csv
